
# Kernel Smoothers

This notebook is intentionally self-contained. It does not import any local utility module, so you can open this file alone during study and see the full statistical workflow, simulation setup, model fitting, evaluation, plotting, and interpretation pattern in one place.



## What problem are we solving?

Kernel smoothing predicts a numeric response by averaging nearby training responses, with nearby defined by a kernel weight function and bandwidth.

**Highest-probability exam pattern:** Implement Nadaraya-Watson smoothing, tune bandwidth by LOOCV, and explain undersmoothing versus oversmoothing.



## Assumptions, intuition, and theory

Small bandwidths use very local data and can be noisy. Large bandwidths average many points and can wash out real curvature. LOOCV estimates how well a bandwidth predicts left-out observations.



    ## Python method notes and documentation

    `np.exp` computes Gaussian kernel weights, weighted averages produce predictions, and LOOCV repeats prediction after removing one observation at a time.

    Documentation links:

    - [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)


## Fully self-contained worked example

Every code line below is commented so the workflow can be scanned quickly under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for bandwidth tuning.
import matplotlib.pyplot as plt  # Import plotting tools for smoother curves.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_sine_regression_data(n=120, noise=0.35, random_state=123):  # Define a nonlinear data simulator for smoothing.
    rng = np.random.default_rng(random_state)  # Create a reproducible random generator.
    x = np.sort(rng.uniform(0.0, 3.0, size=n))  # Draw sorted x values for cleaner curves.
    signal = 4.0 + np.sin(3.0 * x)  # Define the true smooth mean curve.
    y = signal + rng.normal(0.0, noise, size=n)  # Add Gaussian observation noise.
    return x, y, signal  # Return one-dimensional x, response, and true signal.

def nadaraya_watson_predict(x_train, y_train, x_new, bandwidth):  # Define Gaussian-kernel Nadaraya-Watson predictions.
    x_train = np.asarray(x_train).ravel()  # Convert training x values to a flat array.
    y_train = np.asarray(y_train).ravel()  # Convert training y values to a flat array.
    x_new = np.asarray(x_new).ravel()  # Convert prediction points to a flat array.
    predictions = []  # Create a list for predicted values.
    for x0 in x_new:  # Predict one target point at a time.
        weights = np.exp(-0.5 * ((x_train - x0) / bandwidth) ** 2)  # Compute Gaussian weights around x0.
        prediction = np.sum(weights * y_train) / np.sum(weights)  # Compute the weighted average response.
        predictions.append(prediction)  # Store the prediction.
    return np.array(predictions)  # Return predictions as a NumPy array.

def nadaraya_watson_curve(x_train, y_train, bandwidth, num_points=200):  # Define a helper that returns a smooth curve.
    grid = np.linspace(np.min(x_train), np.max(x_train), num_points)  # Build a plotting grid over the observed x range.
    predictions = nadaraya_watson_predict(x_train, y_train, grid, bandwidth)  # Predict the smoother over the grid.
    return grid, predictions  # Return grid and fitted curve.

def kernel_smoother_loocv(x, y, bandwidth_values):  # Define LOOCV bandwidth selection for the kernel smoother.
    x = np.asarray(x).ravel()  # Convert x to a flat array.
    y = np.asarray(y).ravel()  # Convert y to a flat array.
    rows = []  # Create a list for bandwidth results.
    for bandwidth in bandwidth_values:  # Loop over candidate bandwidths.
        squared_errors = []  # Create a list for left-out squared errors.
        for i in range(len(y)):  # Leave out each observation once.
            keep = np.arange(len(y)) != i  # Mark every observation except i as training data.
            prediction = nadaraya_watson_predict(x[keep], y[keep], [x[i]], bandwidth)[0]  # Predict the left-out response.
            squared_errors.append((y[i] - prediction) ** 2)  # Store squared prediction error.
        rows.append({'bandwidth': bandwidth, 'loocv_mse': np.mean(squared_errors)})  # Store average LOOCV MSE for this bandwidth.
    results = pd.DataFrame(rows)  # Convert bandwidth results to a DataFrame.
    best_bandwidth = float(results.loc[results['loocv_mse'].idxmin(), 'bandwidth'])  # Select the bandwidth with smallest LOOCV MSE.
    return results, best_bandwidth  # Return the result table and selected bandwidth.


In [ ]:
x, y, true_signal = make_sine_regression_data(n=90, noise=0.35, random_state=RANDOM_SEED)  # Simulate a one-dimensional smoothing problem.
bandwidth_values = np.linspace(0.05, 0.80, 10)  # Define candidate bandwidths from very local to very smooth.
cv_table, best_bandwidth = kernel_smoother_loocv(x, y, bandwidth_values)  # Choose bandwidth by LOOCV.
grid, fitted_curve = nadaraya_watson_curve(x, y, best_bandwidth)  # Compute the selected smoother curve.
display(cv_table)  # Display the LOOCV table.
print(f'Best bandwidth by LOOCV: {best_bandwidth:.3f}')  # Print the selected bandwidth.
plt.figure(figsize=(6, 4))  # Create a figure for the fitted kernel smoother.
plt.scatter(x, y, s=22, alpha=0.6, label='observed data')  # Plot observed points.
plt.plot(grid, fitted_curve, color='red', label=f'bandwidth={best_bandwidth:.2f}')  # Plot the selected smoother.
plt.xlabel('x')  # Label the predictor axis.
plt.ylabel('y')  # Label the response axis.
plt.title('Kernel smoother selected by LOOCV')  # Title the smoother plot.
plt.legend()  # Show labels.
plt.show()  # Render the smoother plot.
plt.figure(figsize=(6, 4))  # Create a figure for bandwidth tuning.
plt.plot(cv_table['bandwidth'], cv_table['loocv_mse'], marker='o')  # Plot LOOCV MSE against bandwidth.
plt.xlabel('bandwidth')  # Label the tuning axis.
plt.ylabel('LOOCV MSE')  # Label the error axis.
plt.title('Bandwidth controls smoothness')  # Title the tuning plot.
plt.show()  # Render the bandwidth plot.



## Exam-style problem prompt

A prompt gives one numeric predictor and asks for a nonparametric smoother. Write Nadaraya-Watson prediction code and choose the bandwidth with LOOCV.



    ## Adaptable code pattern

    ```python
    # Step 1: Define a grid of candidate bandwidths.
# Step 2: For each bandwidth, leave one observation out.
# Step 3: Predict the left-out response using kernel-weighted averages from the remaining data.
# Step 4: Average squared errors across left-out observations.
# Step 5: Choose the bandwidth with smallest LOOCV MSE.
# Step 6: Plot the selected curve and explain smoothness.
    ```



## What to remember for the exam

The bandwidth is the main tuning parameter. Small means wiggly and high variance; large means smooth and high bias.
